# Active QA — Query Reformulation RL

Train a contextual bandit that decides **when and how to reformulate** a Telugu QA query before retrieval.

**Actions:** KEEP · NORMALIZE · EXPAND · TRANSLITERATE · TRANSLATE · BILINGUALIZE

**Reward:** `α·ΔRecall@k + β·ΔMRR − λ·cost`

## 1. Setup (Kaggle / Colab)

Run this cell first. It installs dependencies and sets up the HF token.

In [ ]:
# Install dependencies
!pip install -q transformers datasets rank-bm25 sentence-transformers torch accelerate

# Verify GPU
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

In [ ]:
# Set HF token (already in config.py, but set as env var too)
import os
import getpass
os.environ['HF_TOKEN'] = getpass.getpass('Enter HF token: ')

## 2. Import Project Modules

Clone or mount this repo, then add to path.

In [ ]:
import sys
sys.path.insert(0, '/content/activeqa')  # adjust path as needed

from config import *
from data.dataset_loader import load_tydiqa, load_indicqa
from data.reformulation_generator import ReformulationGenerator
from retrieval.retriever import BM25Retriever
from retrieval.metrics import recall_at_k, mrr_at_k
from agent.features import QueryFeatureExtractor
from agent.bandit import LinUCB, train_linucb, train_neural_bandit
from agent.policy_network import NeuralBandit
from baselines.baselines import run_all_baselines

## 3. Load Dataset & Build Corpus

In [ ]:
records = load_tydiqa('train')
print(f"Loaded {len(records)} records")
print("Example:", records[0])

In [ ]:
# Build document corpus
corpus = {}
for rec in records:
    for doc_id in rec.get('doc_ids', []):
        corpus[doc_id] = rec.get('gold_answer', '')

retriever = BM25Retriever(corpus)
print(f"Corpus: {len(corpus)} documents")

## 4. Initialize Reformulation Generator

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
reformulator = ReformulationGenerator(device=device)

# Test
test_query = records[0]['query']
print(f"Original: {test_query}")
for action, reformed in reformulator.generate_all(test_query).items():
    print(f"  {action:15s} → {reformed}")

## 5. Compute Reward Matrix

In [ ]:
from train import compute_rewards
rewards = compute_rewards(records, reformulator, retriever)
print(f"Reward matrix: {rewards.shape}")
print("Mean reward per action:")
for a, r in zip(ACTIONS, rewards.mean(axis=0)):
    print(f"  {a:15s} → {r:.4f}")

## 6. Extract Query Features (States)

In [ ]:
extractor = QueryFeatureExtractor(device=device)
states = np.stack([extractor.extract(r['query'], r.get('language', 'telugu'))
                   for r in records])
print(f"State shape: {states.shape}")

## 7. Train LinUCB

In [ ]:
linucb_agent = train_linucb(states, rewards)
print("LinUCB trained.")

## 8. Train Neural Bandit

In [ ]:
neural_agent = NeuralBandit(input_dim=states.shape[1], lr=BANDIT_LR, device=device)
history = train_neural_bandit(neural_agent, states, rewards,
                              epochs=BANDIT_EPOCHS, batch_size=TRAIN_BATCH)

# Plot training curve
import matplotlib.pyplot as plt
plt.plot(history)
plt.xlabel('Epoch')
plt.ylabel('Avg Loss')
plt.title('Neural Bandit Training')
plt.show()

## 9. Evaluate All Agents

In [ ]:
from evaluate import evaluate_linucb, evaluate_neural

print("=== Baselines ===")
baseline_results = run_all_baselines(records, reformulator, retriever)
for r in baseline_results:
    print(f"  {r['action']:15s} — Recall@{TOP_K}: {r['recall']:.4f}  |  MRR: {r['mrr']:.4f}")

print("\n=== RL Agents ===")
res_l = evaluate_linucb(linucb_agent, states, records, reformulator, retriever)
print(f"  LinUCB       — Recall@{TOP_K}: {res_l['recall']:.4f}  |  MRR: {res_l['mrr']:.4f}")

res_n = evaluate_neural(neural_agent, states, records, reformulator, retriever)
print(f"  NeuralBandit — Recall@{TOP_K}: {res_n['recall']:.4f}  |  MRR: {res_n['mrr']:.4f}")

## 10. Save Checkpoints

In [ ]:
import pickle, os
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Save LinUCB
with open(f'{CHECKPOINT_DIR}/linucb.pt', 'wb') as f:
    pickle.dump({'A': linucb_agent.A, 'b': linucb_agent.b,
                 'alpha': linucb_agent.alpha, 'dim': states.shape[1]}, f)

# Save Neural Bandit
neural_agent.save(f'{CHECKPOINT_DIR}/neural_bandit.pt')
print("Checkpoints saved.")